In [ ]:
!pip install -q ultralytics opencv-python

In [ ]:
# laddar upp modellen
from google.colab import files

uploaded = files.upload()
print("Uppladdning klar.")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

# Hitta alla uppladdade modellfiler och videofiler
all_files = os.listdir("/content")
model_candidates = [f for f in all_files if f.endswith(".pt")]
video_candidates = [f for f in all_files if f.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))]

print("Hittade modellfiler:")
for f in model_candidates:
    print("-", f)

print("\nHittade videofiler:")
for f in video_candidates:
    print("-", f)

if not model_candidates:
    raise FileNotFoundError("Ingen .pt-modell hittades i /content")

if not video_candidates:
    raise FileNotFoundError("Ingen videofil hittades i /content")

# Välj modell automatiskt om bara en finns annars skriv in namnet
if len(model_candidates) == 1:
    chosen_model = model_candidates[0]
    print("\nVald modell:", chosen_model)
else:
    chosen_model = input("\nSkriv exakt namn på modellfilen (.pt): ").strip()

# Du väljer själv videon här
chosen_video = input("Skriv exakt namn på videofilen: ").strip()

MODEL_PATH = os.path.join("/content", chosen_model)
VIDEO_PATH = os.path.join("/content", chosen_video)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Modellfilen hittades inte: {MODEL_PATH}")

if not os.path.exists(VIDEO_PATH):
    raise FileNotFoundError(f"Videofilen hittades inte: {VIDEO_PATH}")

print("\nModell:", MODEL_PATH)
print("Video:", VIDEO_PATH)

In [ ]:
# konvertera video till mp4
import subprocess

NORMALIZED_VIDEO = "/content/normalized_input.mp4"

cmd = [
    "ffmpeg",
    "-y",
    "-i", VIDEO_PATH,
    "-vcodec", "libx264",
    "-acodec", "aac",
    "-pix_fmt", "yuv420p",
    NORMALIZED_VIDEO
]

subprocess.run(cmd, check=True)
print("Normaliserad video skapad:", NORMALIZED_VIDEO)

In [ ]:
import cv2
import math
from collections import defaultdict, deque
from ultralytics import YOLO

OUTPUT_PATH = "/content/final_sorted_output.mp4"

# Confidence-gräns för YOLO
CONF_THRESHOLD = 0.35

# Bildstorlek som används vid analys
IMG_SIZE = 960

# Grundgräns för hur nära objekt måste ligga för att räknas som grupp
BASE_GROUP_DISTANCE = 190

# Hur många frames ett tidigare statusvärde får leva kvar
MEMORY_FRAMES = 10

# Hur många tidigare boxar som används för mjukare rörelse
SMOOTH_HISTORY = 5

# Namn i plural för grupper som är rätt sorterade
PLURAL_NAMES = {
    "Dryckeskartong": "KARTONGER",
    "Konservburk": "KONSERVBURKAR",
    "Pantburk": "PANTBURKAR"
}

# Räknar ut mitten av en box
def centroid_from_box(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)

# Räknar avståndet mellan två punkter
def euclidean(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

# Räknar medelvärdet av flera boxar för att få mjukare rörelse
def average_box(boxes):
    n = len(boxes)
    return [
        sum(b[0] for b in boxes) / n,
        sum(b[1] for b in boxes) / n,
        sum(b[2] for b in boxes) / n,
        sum(b[3] for b in boxes) / n,
    ]

# Bestämmer hur stor gruppgränsen ska vara mellan två objekt
def get_dynamic_group_distance(det1, det2):
    x1, y1, x2, y2 = det1["bbox"]
    w1 = x2 - x1
    h1 = y2 - y1

    x1, y1, x2, y2 = det2["bbox"]
    w2 = x2 - x1
    h2 = y2 - y1

    # Tar hänsyn till objektens storlek så att större objekt lättare kan räknas ihop
    avg_size = (w1 + h1 + w2 + h2) / 4

    # Den här versionen är lite mer generös än tidigare
    return max(BASE_GROUP_DISTANCE, avg_size * 1.05)

# Bygger grupper av objekt som ligger nära varandra
def build_groups(detections):
    n = len(detections)
    adj = {i: [] for i in range(n)}

    for i in range(n):
        for j in range(i + 1, n):
            dist = euclidean(detections[i]["centroid"], detections[j]["centroid"])
            threshold = get_dynamic_group_distance(detections[i], detections[j])

            if dist < threshold:
                adj[i].append(j)
                adj[j].append(i)

    visited = set()
    groups = []

    # DFS används för att hitta sammanhängande grupper
    def dfs(node, group):
        visited.add(node)
        group.append(node)
        for nb in adj[node]:
            if nb not in visited:
                dfs(nb, group)

    for i in range(n):
        if i not in visited:
            group = []
            dfs(i, group)
            groups.append(group)

    return groups

# Bestämmer om en grupp är ensam, sorterad eller osorterad
def classify_groups(groups, detections):
    statuses = []

    for idx, group in enumerate(groups):
        class_names = [detections[i]["class_name"] for i in group]
        unique_classes = set(class_names)

        if len(group) == 1:
            status = "alone"
        elif len(unique_classes) == 1:
            status = "sorted"
        else:
            status = "unsorted"

        statuses.append({
            "group_id": idx,
            "members": group,
            "classes": class_names,
            "status": status
        })

    return statuses

# Ritar boxar och text på videobilden
def draw_annotations(frame, detections, group_statuses):
    for group_info in group_statuses:
        members = group_info["members"]
        status = group_info["status"]

        # Väljer färg beroende på status
        if status == "sorted":
            color = (0, 255, 0)      # grön
        elif status == "unsorted":
            color = (0, 0, 255)      # röd
        else:
            color = (0, 255, 255)    # gul

        # Rita en box runt varje objekt i gruppen
        for obj_index in members:
            det = detections[obj_index]
            x1, y1, x2, y2 = map(int, det["bbox"])

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)

            # Vid osorterat skrivs varje objekts namn ut
            if status == "unsorted":
                cv2.putText(
                    frame,
                    det["class_name"],
                    (x1, max(y1 - 10, 25)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    color,
                    2,
                    cv2.LINE_AA
                )

            # Vid ensamt objekt skrivs klassnamn + ensam
            elif status == "alone":
                cv2.putText(
                    frame,
                    f"{det['class_name']} (ensam)",
                    (x1, max(y1 - 10, 25)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    color,
                    2,
                    cv2.LINE_AA
                )

        # Vid sorterad grupp skrivs ett gemensamt namn i plural
        if status == "sorted" and len(members) > 0:
            first_class = detections[members[0]]["class_name"]
            label = PLURAL_NAMES.get(first_class, first_class.upper())

            min_x = min(int(detections[i]["bbox"][0]) for i in members)
            min_y = min(int(detections[i]["bbox"][1]) for i in members)

            cv2.putText(
                frame,
                label,
                (min_x, max(min_y - 15, 25)),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.0,
                color,
                3,
                cv2.LINE_AA
            )

    return frame

In [ ]:
# Laddar YOLO-modellen
model = YOLO(MODEL_PATH)

# Öppnar den konverterade videon
cap = cv2.VideoCapture(NORMALIZED_VIDEO)
if not cap.isOpened():
    raise RuntimeError("Kunde inte öppna videon.")

# Hämtar videons egenskaper
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 25

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Skapar output-video
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

if not out.isOpened():
    raise RuntimeError("Kunde inte skapa output-video.")

# Sparar tidigare boxar för varje track-id för att få mjukare rörelse
track_box_history = defaultdict(lambda: deque(maxlen=SMOOTH_HISTORY))

# Sparar tidigare status en kort stund för att minska flimmer
track_memory = {}

frame_idx = 0

# Läser videon frame för frame
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1

    # Kör tracking så att samma objekt kan följas mellan frames
    results = model.track(
        source=frame,
        conf=CONF_THRESHOLD,
        imgsz=IMG_SIZE,
        persist=True,
        verbose=False
    )

    result = results[0]
    detections = []

    # Om modellen hittade objekt i framen
    if result.boxes is not None and len(result.boxes) > 0:
        boxes_xyxy = result.boxes.xyxy.cpu().tolist()
        classes = result.boxes.cls.cpu().tolist()
        confs = result.boxes.conf.cpu().tolist()

        # Hämtar track-id om de finns
        if result.boxes.id is not None:
            track_ids = result.boxes.id.cpu().tolist()
        else:
            track_ids = [-1] * len(boxes_xyxy)

        # Går igenom alla detekterade objekt
        for box, cls_id, conf, track_id in zip(boxes_xyxy, classes, confs, track_ids):
            cls_id = int(cls_id)
            track_id = int(track_id)
            class_name = model.names[cls_id]

            # Jämnar ut boxens position över flera frames
            if track_id != -1:
                track_box_history[track_id].append(box)
                smooth_box = average_box(list(track_box_history[track_id]))
            else:
                smooth_box = box

            detections.append({
                "track_id": track_id,
                "class_id": cls_id,
                "class_name": class_name,
                "confidence": float(conf),
                "bbox": smooth_box,
                "centroid": centroid_from_box(smooth_box)
            })

    # Bygger grupper av objekt som ligger nära varandra
    groups = build_groups(detections)

    # Bestämmer status för grupperna
    group_statuses = classify_groups(groups, detections)

    # Sparar aktuell status i minnet för varje track-id
    for group_info in group_statuses:
        for member_idx in group_info["members"]:
            det = detections[member_idx]
            tid = det["track_id"]
            if tid != -1:
                track_memory[tid] = {
                    "status": group_info["status"],
                    "last_seen": frame_idx
                }

    # Hämtar tidigare status om objektet nyligen setts
    for det in detections:
        tid = det["track_id"]
        if tid != -1 and tid in track_memory:
            if frame_idx - track_memory[tid]["last_seen"] <= MEMORY_FRAMES:
                det["memory_status"] = track_memory[tid]["status"]
            else:
                det["memory_status"] = None
        else:
            det["memory_status"] = None

    # Om ett objekt tillfälligt blir ensamt kan tidigare status användas en kort stund
    for group_info in group_statuses:
        if group_info["status"] == "alone" and len(group_info["members"]) == 1:
            det = detections[group_info["members"][0]]
            mem = det.get("memory_status")
            if mem in ["sorted", "unsorted"]:
                group_info["status"] = mem

    # Ritar ut resultatet på framen
    annotated = draw_annotations(frame, detections, group_statuses)

    # Sparar framen i output-videon
    out.write(annotated)

# Stänger videon och sparar klart resultatet
cap.release()
out.release()

print("Klart. Sparad video:", OUTPUT_PATH)

In [ ]:
from google.colab import files
files.download(OUTPUT_PATH)